In [0]:
passengers_day1 = [
(101,"Rahul Sharma","Hyderabad","Economy","India"),
(102,"Priya Reddy","Bangalore","Business","India"),
(103,"Amit Kumar","Mumbai","Economy","India"),
(104,"Sneha Patel","Delhi","Premium Economy","India"),
(105,"Farhan Ali","Chennai","Economy","India")
]

columns = [
"passenger_id",
"passenger_name",
"city",
"travel_class",
"country"
]

df_day1 = spark.createDataFrame(passengers_day1, columns)

In [0]:
delta_path = "/tmp/passengers_delta"

df_day1.write \
    .format("delta") \
    .mode("overwrite") \
    .save(delta_path)

---------------------------------------------------------------------------
UnsupportedOperationException             Traceback (most recent call last)
File <command-8108865989868992>, line 6
      1 delta_path = "/tmp/passengers_delta"
      3 df_day1.write \
      4     .format("delta") \
      5     .mode("overwrite") \
----> 6     .save(delta_path)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:703, in DataFrameWriter.save(self, path, format, mode, partitionBy, **options)
    701     self.format(format)
    702 self._write.path = path
--> 703 _, _, ei = self._spark.client.execute_command(
    704     self._write.command(self._spark.client), self._write.observations
    705 )
    706 self._callback(ei)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:1538, in SparkConnectClient.execute_command(self, command, observations, extra_request_metadata)
   1536     req.user_context.user_id = self._user_id
   15

In [0]:
df_day1.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("default.passengers_delta")

In [0]:
spark.sql("SELECT * FROM default.passengers_delta").show()

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         101|  Rahul Sharma|Hyderabad|        Economy|  India|
|         102|   Priya Reddy|Bangalore|       Business|  India|
|         103|    Amit Kumar|   Mumbai|        Economy|  India|
|         104|   Sneha Patel|    Delhi|Premium Economy|  India|
|         105|    Farhan Ali|  Chennai|        Economy|  India|
+------------+--------------+---------+---------------+-------+



In [0]:
passengers_day2 = [
(102,"Priya Reddy","Bangalore","First Class","India"),
(104,"Sneha Patel","Hyderabad","Premium Economy","India"),
(106,"Neha Singh","Pune","Economy","India"),
(107,"Arjun Verma","Kochi","Business","India")
]

df_day2 = spark.createDataFrame(
    passengers_day2,
    columns
)

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(
    spark,
    "default.passengers_delta"
)

In [0]:
delta_table.alias("target") \
.merge(
    df_day2.alias("source"),
    "target.passenger_id = source.passenger_id"
) \
.whenMatchedUpdateAll() \
.whenNotMatchedInsertAll() \
.execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.sql("""
SELECT *
FROM default.passengers_delta
ORDER BY passenger_id
""").show()

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         101|  Rahul Sharma|Hyderabad|        Economy|  India|
|         102|   Priya Reddy|Bangalore|    First Class|  India|
|         103|    Amit Kumar|   Mumbai|        Economy|  India|
|         104|   Sneha Patel|Hyderabad|Premium Economy|  India|
|         105|    Farhan Ali|  Chennai|        Economy|  India|
|         106|    Neha Singh|     Pune|        Economy|  India|
|         107|   Arjun Verma|    Kochi|       Business|  India|
+------------+--------------+---------+---------------+-------+



In [0]:
spark.sql("""
DESCRIBE HISTORY default.passengers_delta
""").show(truncate=False)

+-------+-------------------+---------------+---------------------+---------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+------------------+------------------------------------+------------------------+-----------+-----------------+-------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [0]:
spark.sql("""
SELECT passenger_id,
       passenger_name,
       city,
       travel_class
FROM default.passengers_delta VERSION AS OF 0
WHERE passenger_id = 102
""").show()

+------------+--------------+---------+------------+
|passenger_id|passenger_name|     city|travel_class|
+------------+--------------+---------+------------+
|         102|   Priya Reddy|Bangalore|    Business|
+------------+--------------+---------+------------+



In [0]:
spark.sql("""
SELECT passenger_id,
       passenger_name,
       city,
       travel_class
FROM default.passengers_delta
WHERE passenger_id = 102
""").show()

+------------+--------------+---------+------------+
|passenger_id|passenger_name|     city|travel_class|
+------------+--------------+---------+------------+
|         102|   Priya Reddy|Bangalore| First Class|
+------------+--------------+---------+------------+



In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(
    spark,
    "default.passengers_delta"
)

delta_table.delete(
    "passenger_id = 105"
)

DataFrame[num_affected_rows: bigint]

In [0]:
spark.sql("""
SELECT *
FROM default.passengers_delta
ORDER BY passenger_id
""").show()

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         101|  Rahul Sharma|Hyderabad|        Economy|  India|
|         102|   Priya Reddy|Bangalore|    First Class|  India|
|         103|    Amit Kumar|   Mumbai|        Economy|  India|
|         104|   Sneha Patel|Hyderabad|Premium Economy|  India|
|         106|    Neha Singh|     Pune|        Economy|  India|
|         107|   Arjun Verma|    Kochi|       Business|  India|
+------------+--------------+---------+---------------+-------+



In [0]:
spark.sql("""
DESCRIBE HISTORY default.passengers_delta
""").show(truncate=False)

+-------+-------------------+---------------+---------------------+---------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+------------------+------------------------------------+------------------------+-----------+-----------------+-------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [0]:
%sql 
OPTIMIZE default.passengers_delta
ZORDER BY (city)

path,metrics
abfss://unity-catalog-storage@dbstoragennkcq7nqi5ghc.dfs.core.windows.net/7405619604944460/__unitystorage/catalogs/72978e56-3bda-4cbb-840e-ea4f22a493fa/tables/2601165a-7542-45e3-a23e-adf1e97ba989,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 2038), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1781696174936, 1781696175594, 8, 0, null, List(0, 0), null, 5, 5, 0, 0, null, null)"


In [0]:
%sql
SELECT *
FROM default.passengers_delta
WHERE city = 'Hyderabad'

passenger_id,passenger_name,city,travel_class,country
101,Rahul Sharma,Hyderabad,Economy,India
104,Sneha Patel,Hyderabad,Premium Economy,India


In [0]:
%sql
VACUUM default.passengers_delta DRY RUN

path


In [0]:
%sql
VACUUM default.passengers_delta RETAIN 168 HOURS

path
abfss://unity-catalog-storage@dbstoragennkcq7nqi5ghc.dfs.core.windows.net/7405619604944460/__unitystorage/catalogs/72978e56-3bda-4cbb-840e-ea4f22a493fa/tables/2601165a-7542-45e3-a23e-adf1e97ba989
